# Prepare Data

## Import

In [ ]:
import pickle
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
from imblearn.over_sampling import RandomOverSampler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay , classification_report ,f1_score
from sklearn.model_selection import train_test_split , GridSearchCV
from sklearn.pipeline import make_pipeline

In [ ]:
df = pd.read_csv("data.csv")
df.head()

In [ ]:
df.columns = [c.replace(' ', '_') for c in df.columns]
df.head()

## Explore

In [ ]:
df.info()

In [ ]:
(df.isna().sum() > 0).sum()

## We want to calculate the relative frequencies of the classes

In [ ]:
df['Bankrupt?'].value_counts(normalize= True).plot(kind= 'bar')
plt.xlabel("Bankrupt classes")
plt.ylabel("Frequancy")
plt.title("Class balance");

## Now we show the distributions of the "_Net_Income_to_Total_Assets" column for both groups in the "bankrupt" column

In [ ]:
sns.boxenplot(x="Bankrupt?" , y="_Net_Income_to_Total_Assets" , data=df)
plt.xlabel("Bankrupt classes")
plt.ylabel("Net Income to Total Assets")
plt.title("Distribution of Profit/ Net Income Ratio, by Class");

In [ ]:
df['_Net_Income_to_Total_Assets'].describe()

## We create a histogram to check whether the distrbution is skewed significantly or not

In [ ]:
df["_Net_Income_to_Total_Assets"].hist()
plt.xlabel("Net Income to Total Assets")
plt.ylabel("count")
plt.title("Distrbution of Net Income to Total Assets Ratio");

In [ ]:
q1 , q9 = df['_Net_Income_to_Total_Assets'].quantile([0.1,0.9])
mask = df["_Net_Income_to_Total_Assets"].between(q1 , q9)
sns.boxplot(x='Bankrupt?' , y='_Net_Income_to_Total_Assets', data= df[mask])
plt.xlabel("Bankrupt")
plt.ylabel(" Net Income to Total Assets")
plt.title("Distribution of Net Income to Total Assets Ratio, by Bankruptcy Status");

In [ ]:
df['_Borrowing_dependency'].hist();

In [ ]:
df['_Total_assets_to_GNP_price'].hist();

## Multicollinearity

In [ ]:
corr = df.drop(columns=['Bankrupt?']).corr()
sns.heatmap(corr);

## Split

In [ ]:
target = "Bankrupt?"
X = df.drop(columns=[target])
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , random_state=42)

In [ ]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

## Resample

In [ ]:
over_sampler = RandomOverSampler(random_state=42)
X_train_over , y_train_over = over_sampler.fit_resample(X_train , y_train)
print(X_train_over.shape)
X_train_over.head()

## Build BaseLine

In [ ]:
acc_baseline = y_train.value_counts(normalize=True).max()
print("Baseline Accuracy:", round(acc_baseline, 4))

## iterate

In [ ]:
clf = RandomForestClassifier(random_state=42)

In [ ]:
params= {
    
    "n_estimators":range(25 , 100 , 25),
    "max_depth": range(10 , 70 , 10)
    
}
params

In [ ]:
model = GridSearchCV(

    clf,
    param_grid= params,
    cv=5,
    n_jobs=-1,
    verbose= 1

)
model

In [ ]:
model.fit(X_train_over , y_train_over)

In [ ]:
cv_results = pd.DataFrame(model.cv_results_)
cv_results.sort_values('rank_test_score').head(10)

In [ ]:
model.best_params_

In [ ]:
model.predict(X_train_over)

## Evaluate

In [ ]:
acc_train = model.score(X_train_over , y_train_over)
acc_test = model.score(X_test , y_test)

print(f"Training accuracy: {round(acc_train , 4)}")
print(f"test accuracy: {round(acc_test , 4)}")

## Let's make a confusion matrix to see how our model is making its correct and incorrect predictions.

In [ ]:
ConfusionMatrixDisplay.from_estimator(

    model,
    X_test,
    y_test
    
);

## Let`s make a Classification report to look at the whole picture of the classification model performances. A classification report includes precision, recall, F1 score and support.

In [ ]:
print(classification_report(

    y_test,
    model.predict(X_test)

))

## Communication

In [ ]:
features = X_test.columns
importances = model.best_estimator_.feature_importances_

In [ ]:
feat_imp = pd.Series(importances , index=features).sort_values()
feat_imp.tail().plot(kind= 'barh')
plt.xlabel("Gini Importance")
plt.ylabel("Feature")
plt.title("Feature Importance");

In [ ]:
with open("model-1" , "wb") as f:
    pickle.dump(model ,f)